# Phase 1 — Architecture 3: ALiBi + Multi-Head Latent Attention (decoder-only, from scratch)

**New architectural units (both from Module 2.3):**

| Unit | Replaces | What changes |
|---|---|---|
| **ALiBi** (2.3 #37) | RoPE / position embeddings | No position parameters. Each head adds a fixed bias $-m_h(i-j)$ to the logits, an exponential recency prior with head-specific horizons ($m_h = 2^{-8h/H}$). Relative and translation-invariant, defined for any length. |
| **MLA** (2.3 #57) | MHA K/V projections | Each token is compressed to a latent $c_t\in\mathbb{R}^{128}$, and every head's K, V are reconstructed from it. The KV cache shrinks from 1024 to 128 numbers per token per layer (8×). |

**Why they fit together:** MLA's weight absorption, $q^\top W_{UK} c$, needs nothing position-dependent between $W_Q$ and $W_{UK}$. RoPE puts a rotation there (DeepSeek needs "decoupled RoPE"), while ALiBi's bias sits outside the dot product, so absorption works unchanged. Cell 13 verifies absorbed = materialised numerically.

**Parameter matching:** MLA saves ~0.33M parameters per layer, so this model uses 13 layers (53.9M total) vs the baseline's 12 (54.9M). Everything else (data, tokenizer, optimiser, schedule, tokens/step, validation windows) matches model 01, so differences are attributable to the architecture.

### Pitfalls found in models 01/02 and fixed here

| # | Problem in earlier notebooks | Fix (cell) |
|---|---|---|
| 1 | **Model 01 never applied RoPE**: the registry factory dropped `rope` and `forward` never passed it, so model 01 was effectively a *no-position* (NoPE) model | Positions flow through `forward` (10); test [3] fails if they are not wired in (13) |
| 2 | **Trained in fp32 on the T4**: `is_bf16_supported()` returns True on T4 (emulated), which disabled autocast (~7.9k tok/s) | fp16 autocast + GradScaler on pre-Ampere GPUs (2) |
| 3 | **95/1000 outputs under 128 words** (some 1–30), caused by early EOS | EOS banned until >128 words; generation stops once every row has 128 (12) |
| 4 | ~5 passes over a 150M-token corpus | 300M tokens, same proportions, same val windows (14) |
| 5 | Cosine length guessed (6000), run stopped at ~5800 | `total_steps` from measured throughput (17); final weights evaluated (16) |
| 6 | Init-loss test used unshifted targets (gave 8.9 instead of ~10.4) | Shifted targets (13) |
| 7 | Fully-masked padded query rows could produce NaNs | Padded queries attend to themselves (10) |
| 8 | Inference bugs went unnoticed (cf. the enc-dec submission) | Test [7]: batched generation must equal one-at-a-time generation (13) |
| 9 | Decoding and checkpoint picked blind | Local dev set in the real 512→128-word format with ROUGE-L/BLEU-4 (19) |
| 10 | Approximate per-source val loss; full-prompt logits at prefill; Python-loop repetition penalty | Exact per-source loss (15); last-position logits; vectorised penalty (12) |

### How to run (Kaggle, GPU T4, internet ON, HF secret set)
1. **Run all.** Cell 13 must print `ALL TESTS PASSED` three times before you spend GPU time.
2. Cell 14 encodes 300M tokens (~40 min) unless shards already exist. Saving `/kaggle/working/shards` as a dataset lets other runs skip this.
3. Cell 17 chooses the batch size and schedule (`ARCH` and `TIME_BUDGET_HOURS` live there). Cell 18 trains, and re-running it resumes from `last.pt`.
4. Cells 19–21 select the checkpoint and decoding, write `submission.zip`, and print the report analyses.

**Same notebook, other runs:** `ARCH = "alibi_mha"` gives the ALiBi-only ablation (separates ALiBi's effect from MLA's), and `ARCH = "rope_mha"` gives the corrected RoPE baseline.

In [1]:
# Cell 0 — Dependencies
!pip install -q ijson

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 149.9/149.9 kB 3.8 MB/s eta 0:00:00


In [2]:
# Cell 1 — Hugging Face login (token from Kaggle secrets)
import os
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient


try:
    user_secrets = UserSecretsClient()
    secret_value_0 = user_secrets.get_secret("HF")
    # secret_value = UserSecretsClient().get_secret("HF")
    os.environ["HF_TOKEN"] = secret_value_0
    login(secret_value_0)
    print("HF login ok")
except Exception as e:
    print("HF secret not available:", e)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


HF login ok


In [3]:
# Cell 2 — Imports, paths, device, PRECISION, reproducibility

import os, sys, json, math, time, random, hashlib, gc, shutil, zipfile
from pathlib import Path
from dataclasses import dataclass, asdict, field
from typing import Optional, Dict, List, Tuple, Any

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

IS_KAGGLE = Path("/kaggle/working").exists()
WORK_DIR = Path("/kaggle/working") if IS_KAGGLE else Path("./work")
WORK_DIR.mkdir(parents=True, exist_ok=True)
TOKENIZER_DIR = Path("/kaggle/input/datasets/ascyrax/capstone-tokenizer-large") if IS_KAGGLE \
    else Path("./input")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# FIX: model 01 used `USE_FP16 = not torch.cuda.is_bf16_supported()`. On a T4 that function
# returns True (bf16 is *emulated*), so autocast was disabled and the whole run was in fp32
# (~7.9k tok/s). T4 / P100 (compute capability < 8.0) have fast fp16 but no native bf16, so:
#   capability >= 8.0 (A100, L4, ...) -> bf16 autocast, no loss scaling needed
#   capability <  8.0 (T4, P100)      -> fp16 autocast + GradScaler
if device.type == "cuda":
    cap = torch.cuda.get_device_capability(0)
    AMP_DTYPE = torch.bfloat16 if cap[0] >= 8 else torch.float16
    USE_AMP = True
    print("GPU:", torch.cuda.get_device_name(0), "| capability", cap)
else:
    AMP_DTYPE, USE_AMP = torch.float32, False
USE_SCALER = USE_AMP and AMP_DTYPE == torch.float16
print(f"device={device} | AMP dtype={AMP_DTYPE} | grad scaler={USE_SCALER}")
print("torch", torch.__version__)

SEED = 1337
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if device.type == "cuda":
    torch.cuda.manual_seed_all(SEED)
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

GPU: Tesla T4 | capability (7, 5)
device=cuda | AMP dtype=torch.float16 | grad scaler=True
torch 2.11.0+cu128


In [4]:
# Cell 3 — The shared byte-level BPE tokenizer (same as every other model)

if str(TOKENIZER_DIR) not in sys.path:
    sys.path.append(str(TOKENIZER_DIR))
from bpe_tokenizer import BPETokenizer, load_tokenizer

tokenizer = load_tokenizer(str(TOKENIZER_DIR))
VOCAB_SIZE = len(tokenizer.id_to_bytes)
BOS_ID, EOS_ID = tokenizer.bos_id, tokenizer.eos_id
PAD_ID, UNK_ID = tokenizer.pad_id, tokenizer.unk_id
print("vocab_size:", VOCAB_SIZE, "| BOS", BOS_ID, "EOS", EOS_ID, "PAD", PAD_ID, "UNK", UNK_ID)

_s = "The Supreme Court of India held that the appeal must be allowed."
assert tokenizer.decode(tokenizer.encode(_s, allow_special=False)) == _s, "round-trip failed"
print("round-trip ok")
# Reference numbers from the tokenizer report: 1.419 tokens/word overall (1.30-1.47 per source).
# 512-word prompt ~ 727 tokens; 128-word continuation ~ 182 tokens; together ~ 910 < 1024. Good.

vocab_size: 32000 | BOS 2 EOS 3 PAD 0 UNK 1
round-trip ok


In [5]:
# Guard against O(L^2) BPE on pathological chunks (long runs of dashes, dots, underscores,
# spaces). Chunks longer than 1000 bytes are BPE-encoded in 1000-byte pieces. Normal text
# never has chunks this long, so its tokenization is unchanged. Byte-level BPE decodes the
# pieces back to the exact same bytes.
_MAX_PRETOK_BYTES = 1000
_orig_encode_pretok = tokenizer._encode_pretok_bytes

def _encode_pretok_bytes_capped(bs):
    if len(bs) <= _MAX_PRETOK_BYTES:
        return _orig_encode_pretok(bs)
    out = []
    for i in range(0, len(bs), _MAX_PRETOK_BYTES):
        out.extend(_orig_encode_pretok(bs[i:i + _MAX_PRETOK_BYTES]))
    return out

tokenizer._encode_pretok_bytes = _encode_pretok_bytes_capped

_t = time.time()
_ids = tokenizer.encode("-" * 200_000, allow_special=False)
assert tokenizer.decode(_ids) == "-" * 200_000
print(f"long-run guard ok: 200k-char run encoded in {time.time() - _t:.1f}s")

long-run guard ok: 200k-char run encoded in 0.1s


## Model
All components are hand-written; PyTorch supplies tensors, autograd, `nn.Linear`/`nn.Embedding`, SDPA and the optimiser.

In [6]:
# Cell 4 — Model configuration
#
# One config drives three architectures (all registered, all testable):
#   preset_alibi_mla()  -> the NEW architecture (ALiBi positions + Multi-Head Latent Attention)
#   preset_alibi_mha()  -> ablation: ALiBi with ordinary MHA (isolates the ALiBi effect)
#   preset_rope_mha()   -> corrected baseline: RoPE + MHA (model 01 accidentally had NO positions)

@dataclass
class ModelConfig:
    # --- shape ---
    vocab_size: int = 32000
    n_layer: int = 13
    d_model: int = 512
    n_head: int = 8
    n_kv_head: int = 8                # only used by mixer_type="attention" (GQA if < n_head)
    head_dim: int = 64
    ff_mult: float = 8/3              # SwiGLU hidden = round64(ff_mult * d_model) = 1408
    d_ff: Optional[int] = None
    max_seq_len: int = 1024           # TRAINING context. ALiBi itself has no length limit.

    # --- components ---
    norm_type: str = "rmsnorm"        # "rmsnorm" | "layernorm"
    norm_eps: float = 1e-5
    pos_type: str = "alibi"           # "alibi" | "rope" | "learned" | "none"
    rope_theta: float = 10000.0
    mixer_type: str = "mla"           # "attention" (MHA/GQA) | "mla"
    kv_latent_dim: int = 128          # d_c: MLA caches only this many numbers per token per layer
    mla_latent_norm: bool = True      # RMSNorm on the latent c_t (as in DeepSeek-V2)
    mlp_type: str = "swiglu"          # "swiglu" | "gelu"
    channel_mixer_type: str = "mlp"

    # --- misc ---
    bias: bool = False
    attn_dropout: float = 0.0
    resid_dropout: float = 0.0
    embd_dropout: float = 0.0
    tie_embeddings: bool = True
    init_std: float = 0.02
    use_sdpa: bool = True

    def __post_init__(self):
        assert self.d_model == self.n_head * self.head_dim, "d_model must equal n_head*head_dim"
        assert self.n_head % self.n_kv_head == 0, "n_head must be divisible by n_kv_head"
        assert self.pos_type in ("alibi", "rope", "learned", "none")
        assert self.mixer_type in ("attention", "mla")
        if self.mixer_type == "mla":
            # RoPE rotates q and k by a position-dependent matrix that sits BETWEEN W_Q and
            # W_UK, which breaks MLA's weight absorption (DeepSeek needs "decoupled RoPE").
            # ALiBi only adds a bias outside q.k, so it composes cleanly with MLA.
            assert self.pos_type != "rope", "MLA + RoPE needs decoupled RoPE; use ALiBi"
            assert not self.bias, "MLA absorption assumes bias-free up-projections"
        if self.d_ff is None:
            raw = int(self.ff_mult * self.d_model)
            self.d_ff = 64 * ((raw + 63) // 64)


def preset_alibi_mla(vocab_size=None, max_seq_len=1024):
    # MLA saves ~0.33M attention params per layer vs MHA (d_c=128). 13 layers instead of 12
    # reinvests that saving so the model stays parameter-matched to the 12-layer baseline.
    return ModelConfig(vocab_size=vocab_size or VOCAB_SIZE, n_layer=13, pos_type="alibi",
                       mixer_type="mla", kv_latent_dim=128, max_seq_len=max_seq_len)

def preset_alibi_mha(vocab_size=None, max_seq_len=1024):
    return ModelConfig(vocab_size=vocab_size or VOCAB_SIZE, n_layer=12, pos_type="alibi",
                       mixer_type="attention", max_seq_len=max_seq_len)

def preset_rope_mha(vocab_size=None, max_seq_len=1024):
    return ModelConfig(vocab_size=vocab_size or VOCAB_SIZE, n_layer=12, pos_type="rope",
                       mixer_type="attention", max_seq_len=max_seq_len)

PRESETS = {"alibi_mla": preset_alibi_mla, "alibi_mha": preset_alibi_mha, "rope_mha": preset_rope_mha}
print(json.dumps(asdict(preset_alibi_mla()), indent=2))

{
  "vocab_size": 32000,
  "n_layer": 13,
  "d_model": 512,
  "n_head": 8,
  "n_kv_head": 8,
  "head_dim": 64,
  "ff_mult": 2.6666666666666665,
  "d_ff": 1408,
  "max_seq_len": 1024,
  "norm_type": "rmsnorm",
  "norm_eps": 1e-05,
  "pos_type": "alibi",
  "rope_theta": 10000.0,
  "mixer_type": "mla",
  "kv_latent_dim": 128,
  "mla_latent_norm": true,
  "mlp_type": "swiglu",
  "channel_mixer_type": "mlp",
  "bias": false,
  "attn_dropout": 0.0,
  "resid_dropout": 0.0,
  "embd_dropout": 0.0,
  "tie_embeddings": true,
  "init_std": 0.02,
  "use_sdpa": true
}


In [7]:
# Cell 5 — Normalisation layers (hand-written)

class LayerNorm(nn.Module):
    """Classic LayerNorm: centre by the mean, scale by the std, learn gain g and bias b."""
    def __init__(self, d: int, eps: float = 1e-5):
        super().__init__()
        self.g = nn.Parameter(torch.ones(d))
        self.b = nn.Parameter(torch.zeros(d))
        self.eps = eps

    def forward(self, x):
        xf = x.float()
        mean = xf.mean(-1, keepdim=True)
        var = xf.var(-1, keepdim=True, unbiased=False)
        return (((xf - mean) / torch.sqrt(var + self.eps)).to(x.dtype)) * self.g + self.b


class RMSNorm(nn.Module):
    """RMSNorm: no mean-centring and no bias; x / rms(x) * g. Statistics computed in fp32."""
    def __init__(self, d: int, eps: float = 1e-5):
        super().__init__()
        self.g = nn.Parameter(torch.ones(d))
        self.eps = eps

    def forward(self, x):
        rms = torch.sqrt(x.float().pow(2).mean(-1, keepdim=True) + self.eps)
        return (x.float() / rms).to(x.dtype) * self.g


def build_norm(cfg: ModelConfig, d: Optional[int] = None):
    d = d or cfg.d_model
    if cfg.norm_type == "layernorm":
        return LayerNorm(d, cfg.norm_eps)
    if cfg.norm_type == "rmsnorm":
        return RMSNorm(d, cfg.norm_eps)
    raise ValueError(cfg.norm_type)

In [8]:
# Cell 6 — Positional mechanisms: ALiBi (new) and RoPE (fixed wiring)
#
# ALiBi (Press et al., 2022). No position embeddings at all. Each head h adds a fixed bias
#       s_ij = q_i.k_j / sqrt(d)  -  m_h * (i - j),        j <= i
# Inside the softmax this multiplies attention by exp(-m_h (i-j)): an exponential recency prior.
# Only the distance (i - j) matters: softmax ignores a per-row constant, and
# -m(i-j) = -m*i + m*j, where -m*i is constant along row i.
# Slopes are geometric, m_h = 2^(-8h/H), h = 1..H. For H = 8: 1/2, 1/4, ..., 1/256, so heads
# range from "looks only a few tokens back" to "looks hundreds of tokens back".

def alibi_slopes(n_head: int) -> List[float]:
    def pow2_slopes(n):
        start = 2.0 ** (-8.0 / n)
        return [start ** (i + 1) for i in range(n)]
    if math.log2(n_head).is_integer():
        return pow2_slopes(n_head)
    # Non-power-of-2 head counts: the paper's interpolation rule.
    c = 2 ** math.floor(math.log2(n_head))
    return pow2_slopes(c) + pow2_slopes(2 * c)[0::2][: n_head - c]


class ALiBi(nn.Module):
    def __init__(self, n_head: int):
        super().__init__()
        self.register_buffer("slopes", torch.tensor(alibi_slopes(n_head), dtype=torch.float32),
                             persistent=False)

    def bias(self, q_len: int, k_len: int, device) -> torch.Tensor:
        """[1, H, q_len, k_len] float32. Queries are the LAST q_len of the k_len positions
        (so it also works for KV-cached decoding). Includes the causal mask (-inf for j > i)."""
        q_pos = torch.arange(k_len - q_len, k_len, device=device)
        k_pos = torch.arange(k_len, device=device)
        dist = (q_pos[:, None] - k_pos[None, :]).float()          # i - j
        b = -self.slopes.to(device)[:, None, None] * dist          # [H, Tq, Tk]
        b = b.masked_fill(dist < 0, float("-inf"))
        return b.unsqueeze(0)


class RoPE(nn.Module):
    """Rotary embeddings. Table is built for 2x the training length so generation never overflows."""
    def __init__(self, head_dim: int, max_pos: int, theta: float = 10000.0):
        super().__init__()
        assert head_dim % 2 == 0
        inv_freq = 1.0 / (theta ** (torch.arange(0, head_dim, 2).float() / head_dim))
        freqs = torch.outer(torch.arange(max_pos).float(), inv_freq)   # [P, D/2]
        self.register_buffer("cos", freqs.cos(), persistent=False)
        self.register_buffer("sin", freqs.sin(), persistent=False)

    def tables(self, position_ids: torch.Tensor):
        """position_ids [B, T] -> (cos, sin) each [B, 1, T, D/2]."""
        return self.cos[position_ids].unsqueeze(1), self.sin[position_ids].unsqueeze(1)

    @staticmethod
    def apply(x, cos, sin):
        """x [B, H, T, D]; rotation done in fp32 for stability."""
        dt = x.dtype
        x = x.float()
        x1, x2 = x[..., 0::2], x[..., 1::2]
        out = torch.stack([x1 * cos - x2 * sin, x1 * sin + x2 * cos], dim=-1).flatten(-2)
        return out.to(dt)

In [9]:
# Cell 7 — Shared attention kernel + standard MHA/GQA token mixer
#
# The Backbone builds ONE additive mask per forward pass (`attn_bias`, shape [B or 1, H or 1, T, S])
# containing ALiBi + causal + padding, and passes it to every layer. Every token mixer then
# calls `attend`. When attn_bias is None we use SDPA's fast is_causal path.

def attend(q, k, v, attn_bias, is_causal, scale, dropout_p=0.0, use_sdpa=True):
    """q [B,H,T,Dq], k [B,H,S,Dq], v [B,H,S,Dv] -> [B,H,T,Dv]."""
    if attn_bias is not None:
        attn_bias = attn_bias.to(q.dtype)          # SDPA requires mask dtype == query dtype
        is_causal = False
    if use_sdpa:
        return F.scaled_dot_product_attention(q, k, v, attn_mask=attn_bias,
                                              dropout_p=dropout_p, is_causal=is_causal,
                                              scale=scale)
    # Manual reference path (used by the tests to cross-check SDPA).
    scores = (q.float() @ k.float().transpose(-2, -1)) * scale
    if is_causal:
        T, S = q.shape[-2], k.shape[-2]
        scores = scores.masked_fill(
            torch.triu(torch.ones(T, S, dtype=torch.bool, device=q.device), diagonal=S - T + 1),
            float("-inf"))
    if attn_bias is not None:
        scores = scores + attn_bias.float()
    p = F.softmax(scores, dim=-1)
    if dropout_p > 0:
        p = F.dropout(p, dropout_p)
    return (p @ v.float()).to(q.dtype)


class Attention(nn.Module):
    """Multi-head (n_kv_head == n_head) or grouped-query (n_kv_head < n_head) self-attention.
    Cache format: (k, v), each [B, n_kv_head, S, head_dim]."""
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        self.cfg = cfg
        self.n_head, self.n_kv_head, self.head_dim = cfg.n_head, cfg.n_kv_head, cfg.head_dim
        self.scale = 1.0 / math.sqrt(cfg.head_dim)
        self.q_proj = nn.Linear(cfg.d_model, cfg.n_head * cfg.head_dim, bias=cfg.bias)
        self.k_proj = nn.Linear(cfg.d_model, cfg.n_kv_head * cfg.head_dim, bias=cfg.bias)
        self.v_proj = nn.Linear(cfg.d_model, cfg.n_kv_head * cfg.head_dim, bias=cfg.bias)
        self.o_proj = nn.Linear(cfg.n_head * cfg.head_dim, cfg.d_model, bias=cfg.bias)
        self.resid_dropout = nn.Dropout(cfg.resid_dropout)

    def _split(self, x, n):
        B, T, _ = x.shape
        return x.view(B, T, n, self.head_dim).transpose(1, 2)

    @staticmethod
    def cache_len(cache):
        return cache[0].shape[2]

    def cache_numbers_per_token(self):
        return 2 * self.n_kv_head * self.head_dim

    def forward(self, x, attn_bias=None, rope_cs=None, kv_cache=None):
        B, T, _ = x.shape
        q = self._split(self.q_proj(x), self.n_head)
        k = self._split(self.k_proj(x), self.n_kv_head)
        v = self._split(self.v_proj(x), self.n_kv_head)
        if rope_cs is not None:                       # FIX: RoPE is now actually applied
            cos, sin = rope_cs
            q, k = RoPE.apply(q, cos, sin), RoPE.apply(k, cos, sin)
        if kv_cache is not None:
            k = torch.cat([kv_cache[0], k], dim=2)
            v = torch.cat([kv_cache[1], v], dim=2)
        new_cache = (k, v)
        if self.n_kv_head != self.n_head:
            rep = self.n_head // self.n_kv_head
            k, v = k.repeat_interleave(rep, dim=1), v.repeat_interleave(rep, dim=1)
        out = attend(q, k, v, attn_bias, is_causal=(attn_bias is None and T > 1),
                     scale=self.scale,
                     dropout_p=self.cfg.attn_dropout if self.training else 0.0,
                     use_sdpa=self.cfg.use_sdpa)
        out = out.transpose(1, 2).contiguous().view(B, T, -1)
        return self.resid_dropout(self.o_proj(out)), new_cache

In [10]:
# Cell 8 — Multi-Head Latent Attention (MLA, DeepSeek-V2), adapted to ALiBi
#
#   c_t      = Norm(W_DKV x_t)          in R^{d_c}        (the ONLY thing we cache)
#   k_t^(h)  = W_UK^(h) c_t,   v_t^(h) = W_UV^(h) c_t    (every head gets its own K, V)
#   q_t^(h)  = W_Q^(h) x_t
#
# KV cache per token per layer: MHA stores 2*H*D_h = 1024 numbers; MLA stores d_c = 128 (8x less).
#
# MLA vs GQA: GQA *ties* heads (groups share identical K,V). MLA gives each head distinct K,V,
# all reconstructed from a shared rank-d_c subspace. With d_c = 2*n_kv*D_h, MLA can exactly
# express GQA (the up-projections just copy slices), so at equal cache size GQA is a special case.
#
# Weight absorption (inference trick, implemented below as `absorb=True`):
#   q_i^(h).k_j^(h) = (W_Q^(h) x_i)^T W_UK^(h) c_j = [W_UK^(h)T q_i^(h)] . c_j
# so we can attend directly against the cached latents without ever building K:
#   q_lat = W_UK^T q,  scores = q_lat . c,  out_lat = softmax(scores) c,  out^(h) = W_UV^(h) out_lat
# RoPE would insert a position-dependent rotation R_{i-j} between W_Q and W_UK and break this;
# ALiBi's bias lives outside the dot product, so absorption is unaffected.
# Training uses the materialised path (simpler, identical maths); a test checks both agree.

class MLAttention(nn.Module):
    """Cache format: c, [B, S, d_c]."""
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        self.cfg = cfg
        H, Dh, dc = cfg.n_head, cfg.head_dim, cfg.kv_latent_dim
        self.n_head, self.head_dim, self.d_c = H, Dh, dc
        self.scale = 1.0 / math.sqrt(Dh)              # scores are still per-head q.k with dim D_h
        self.q_proj = nn.Linear(cfg.d_model, H * Dh, bias=False)
        self.kv_down = nn.Linear(cfg.d_model, dc, bias=False)          # W_DKV
        self.kv_norm = build_norm(cfg, dc) if cfg.mla_latent_norm else nn.Identity()
        self.k_up = nn.Linear(dc, H * Dh, bias=False)                  # W_UK (all heads)
        self.v_up = nn.Linear(dc, H * Dh, bias=False)                  # W_UV (all heads)
        self.o_proj = nn.Linear(H * Dh, cfg.d_model, bias=False)
        self.resid_dropout = nn.Dropout(cfg.resid_dropout)
        self.absorb = False                           # runtime switch (inference only)

    @staticmethod
    def cache_len(cache):
        return cache.shape[1]

    def cache_numbers_per_token(self):
        return self.d_c

    def forward(self, x, attn_bias=None, rope_cs=None, kv_cache=None):
        assert rope_cs is None, "MLA is used with ALiBi (or no positions), not RoPE"
        B, T, _ = x.shape
        H, Dh = self.n_head, self.head_dim
        c = self.kv_norm(self.kv_down(x))                         # [B, T, d_c]
        if kv_cache is not None:
            c = torch.cat([kv_cache, c], dim=1)                   # [B, S, d_c]
        new_cache = c
        S = c.shape[1]
        q = self.q_proj(x).view(B, T, H, Dh).transpose(1, 2)      # [B, H, T, Dh]
        is_causal = (attn_bias is None and T > 1)
        dp = self.cfg.attn_dropout if self.training else 0.0

        if not self.absorb:
            k = self.k_up(c).view(B, S, H, Dh).transpose(1, 2)    # [B, H, S, Dh]
            v = self.v_up(c).view(B, S, H, Dh).transpose(1, 2)
            out = attend(q, k, v, attn_bias, is_causal, self.scale, dp, self.cfg.use_sdpa)
        else:
            W_uk = self.k_up.weight.view(H, Dh, self.d_c)         # rows of head h -> W_UK^(h)
            W_uv = self.v_up.weight.view(H, Dh, self.d_c)
            q_lat = torch.einsum("bhtd,hdc->bhtc", q, W_uk.to(q.dtype))   # W_UK^T q
            c_h = c.unsqueeze(1).expand(B, H, S, self.d_c)
            out_lat = attend(q_lat, c_h, c_h, attn_bias, is_causal, self.scale, dp,
                             self.cfg.use_sdpa)                   # [B, H, T, d_c]
            out = torch.einsum("bhtc,hdc->bhtd", out_lat, W_uv.to(out_lat.dtype))

        out = out.transpose(1, 2).contiguous().view(B, T, H * Dh)
        return self.resid_dropout(self.o_proj(out)), new_cache

In [11]:
# Cell 9 — Channel mixers, registries, transformer block

class SwiGLU(nn.Module):
    """down( silu(gate(x)) * up(x) )."""
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        self.gate_proj = nn.Linear(cfg.d_model, cfg.d_ff, bias=cfg.bias)
        self.up_proj = nn.Linear(cfg.d_model, cfg.d_ff, bias=cfg.bias)
        self.down_proj = nn.Linear(cfg.d_ff, cfg.d_model, bias=cfg.bias)
        self.drop = nn.Dropout(cfg.resid_dropout)

    def forward(self, x):
        return self.drop(self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x)))


class GELUMLP(nn.Module):
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        self.fc1 = nn.Linear(cfg.d_model, cfg.d_ff, bias=cfg.bias)
        self.down_proj = nn.Linear(cfg.d_ff, cfg.d_model, bias=cfg.bias)
        self.drop = nn.Dropout(cfg.resid_dropout)

    def forward(self, x):
        return self.drop(self.down_proj(F.gelu(self.fc1(x))))


TOKEN_MIXERS: Dict[str, Any] = {}
CHANNEL_MIXERS: Dict[str, Any] = {}

def register_token_mixer(name):
    def deco(fn):
        TOKEN_MIXERS[name] = fn
        return fn
    return deco

def register_channel_mixer(name):
    def deco(fn):
        CHANNEL_MIXERS[name] = fn
        return fn
    return deco

# FIX: token mixers no longer receive `rope` at build time (model 01 silently dropped it there).
# Positional information now flows through forward(): rope_cs / attn_bias, built in the Backbone.
@register_token_mixer("attention")
def _build_attention(cfg):
    return Attention(cfg)

@register_token_mixer("mla")
def _build_mla(cfg):
    return MLAttention(cfg)

@register_channel_mixer("mlp")
def _build_mlp(cfg):
    return {"swiglu": SwiGLU, "gelu": GELUMLP}[cfg.mlp_type](cfg)


class Block(nn.Module):
    """Pre-norm block: x = x + mixer(norm(x)); x = x + mlp(norm(x))."""
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        self.norm1 = build_norm(cfg)
        self.mixer = TOKEN_MIXERS[cfg.mixer_type](cfg)
        self.norm2 = build_norm(cfg)
        self.cmix = CHANNEL_MIXERS[cfg.channel_mixer_type](cfg)

    def forward(self, x, **kw):
        h, new_cache = self.mixer(self.norm1(x), **kw)
        x = x + h
        x = x + self.cmix(self.norm2(x))
        return x, new_cache

In [12]:
# Cell 10 — Backbone and language model

class Backbone(nn.Module):
    """Embedding -> N blocks -> final norm. Builds positional info and masks ONCE per forward."""
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.emb_drop = nn.Dropout(cfg.embd_dropout)
        self.alibi = ALiBi(cfg.n_head) if cfg.pos_type == "alibi" else None
        self.rope = RoPE(cfg.head_dim, 2 * cfg.max_seq_len, cfg.rope_theta) \
            if cfg.pos_type == "rope" else None
        self.pos_emb = nn.Embedding(2 * cfg.max_seq_len, cfg.d_model) \
            if cfg.pos_type == "learned" else None
        self.blocks = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layer)])
        self.norm_f = build_norm(cfg)

    def _attn_bias(self, B, T, past, attention_mask, device):
        """Additive mask [B|1, H|1, T, S] or None (=> SDPA fast causal path)."""
        S = past + T
        need = (self.alibi is not None) or (attention_mask is not None) or (past > 0 and T > 1)
        if not need:
            return None
        if self.alibi is not None:
            bias = self.alibi.bias(T, S, device)                          # ALiBi + causal
        else:
            q_pos = torch.arange(past, S, device=device)
            k_pos = torch.arange(S, device=device)
            bias = torch.zeros(T, S, device=device).masked_fill(
                k_pos[None, :] > q_pos[:, None], float("-inf"))[None, None]
        if attention_mask is not None:
            # Mask padded KEYS. A padded QUERY would then have every key masked -> softmax of
            # all -inf -> NaN, which can leak into later layers through 0*NaN. So let each
            # padded query attend to itself (its output is never used).
            key_pad = ~attention_mask.bool()                                   # [B, S]
            q_idx = torch.arange(past, S, device=device)
            k_idx = torch.arange(S, device=device)
            is_self = q_idx[:, None] == k_idx[None, :]                         # [T, S]
            block = key_pad[:, None, :] & ~is_self[None]                       # [B, T, S]
            pad = torch.zeros(B, 1, T, S, device=device).masked_fill(block[:, None], float("-inf"))
            bias = bias + pad
        return bias

    def forward(self, input_ids, attention_mask=None, kv_caches=None):
        B, T = input_ids.shape
        dev = input_ids.device
        past = 0 if kv_caches is None else self.blocks[0].mixer.cache_len(kv_caches[0])
        x = self.tok_emb(input_ids)

        rope_cs = None
        if self.rope is not None or self.pos_emb is not None:
            if attention_mask is not None:       # left padding: real tokens start at position 0
                pos = (attention_mask.long().cumsum(-1) - 1).clamp(min=0)[:, -T:]
            else:
                pos = torch.arange(past, past + T, device=dev).unsqueeze(0).expand(B, T)
            if self.rope is not None:
                rope_cs = self.rope.tables(pos)
            else:
                x = x + self.pos_emb(pos)

        attn_bias = self._attn_bias(B, T, past, attention_mask, dev)
        x = self.emb_drop(x)
        new_caches = []
        for i, blk in enumerate(self.blocks):
            x, nc = blk(x, attn_bias=attn_bias, rope_cs=rope_cs,
                        kv_cache=None if kv_caches is None else kv_caches[i])
            new_caches.append(nc)
        return self.norm_f(x), new_caches


class LegalLM(nn.Module):
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        self.cfg = cfg
        self.backbone = Backbone(cfg)
        self.lm_head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        if cfg.tie_embeddings:
            self.lm_head.weight = self.backbone.tok_emb.weight

    def set_mla_absorb(self, flag: bool):
        for blk in self.backbone.blocks:
            if isinstance(blk.mixer, MLAttention):
                blk.mixer.absorb = flag

    def forward(self, input_ids, targets=None, attention_mask=None, kv_caches=None,
                last_only=False, reduction="mean"):
        h, new_caches = self.backbone(input_ids, attention_mask=attention_mask,
                                      kv_caches=kv_caches)
        if last_only:                 # generation: only the last position's logits are needed
            h = h[:, -1:, :]
        logits = self.lm_head(h)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.float().view(-1, logits.size(-1)), targets.reshape(-1),
                                   ignore_index=-100, reduction=reduction)
            if reduction == "none":
                loss = loss.view(targets.shape)
        return logits, loss, new_caches

In [13]:
# Cell 11 — Init, parameter counting, optimiser groups, checkpoint I/O

def init_weights(model: LegalLM, cfg: ModelConfig):
    """N(0, 0.02) everywhere; residual-branch output projections scaled by 1/sqrt(2L)
    so the residual stream's variance does not grow with depth."""
    for m in model.modules():
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, 0.0, cfg.init_std)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, 0.0, cfg.init_std)
    scale = 1.0 / math.sqrt(2 * cfg.n_layer)
    with torch.no_grad():
        for blk in model.backbone.blocks:
            blk.mixer.o_proj.weight.mul_(scale)
            blk.cmix.down_proj.weight.mul_(scale)


def count_params(model: LegalLM) -> Dict[str, int]:
    seen, total = set(), 0
    for p in model.parameters():
        if id(p) not in seen:
            seen.add(id(p))
            total += p.numel()
    emb = model.backbone.tok_emb.weight.numel()
    attn = sum(p.numel() for blk in model.backbone.blocks for p in blk.mixer.parameters())
    return {"total": total, "embedding": emb, "non_embedding": total - emb, "attention": attn}


def build_param_groups(model: nn.Module, weight_decay: float):
    """Decay only 2-D weight matrices (not embeddings, norm gains or biases)."""
    decay, no_decay = [], []
    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue
        if p.ndim >= 2 and "tok_emb" not in name and "pos_emb" not in name:
            decay.append(p)
        else:
            no_decay.append(p)
    return [{"params": decay, "weight_decay": weight_decay},
            {"params": no_decay, "weight_decay": 0.0}]


def save_model(path, model: LegalLM, cfg: ModelConfig, extra: dict = None):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    torch.save({"model": model.state_dict(), "config": asdict(cfg), "extra": extra or {}}, path)


def load_model(path, map_location="cpu") -> Tuple[LegalLM, ModelConfig, dict]:
    ckpt = torch.load(path, map_location=map_location, weights_only=False)
    cfg = ModelConfig(**ckpt["config"])
    model = LegalLM(cfg)
    model.load_state_dict(ckpt["model"])
    return model, cfg, ckpt.get("extra", {})

## Generation and tests
Run the tests **before** training. They take about a minute and cover positions, causality, KV cache, padding, batched-vs-single generation, MLA absorption and ALiBi arithmetic.

In [14]:
# Cell 12 — Generation (word-aware, matches the task: 512-word prompt -> 128-word continuation)
#
# Fixes vs model 01:
#  * EOS (and BOS/PAD/UNK) are banned until a row has produced >128 words. Model 01 emitted
#    EOS early on 95/1000 prompts (some outputs were 1-30 words), which hurts BLEU/ROUGE/BERTScore.
#  * Stops as soon as every row has its 128 words (faster), instead of always running N tokens.
#  * Prefill computes logits for the last position only (no [B, 768, 32000] tensor).
#  * Repetition penalty vectorised; optional no-repeat-n-gram blocking.

def _count_words(ids: List[int]) -> int:
    return len(tokenizer.decode(ids).split())

@torch.no_grad()
def generate(model: LegalLM, prompts: List[List[int]], max_new_tokens: int = 256,
             temperature: float = 0.0, top_k: int = 0, top_p: float = 1.0,
             repetition_penalty: float = 1.0, no_repeat_ngram: int = 0,
             min_words: int = 128, check_every: int = 8, use_amp: bool = True) -> List[List[int]]:
    model.eval()
    dev = next(model.parameters()).device
    B, L = len(prompts), max(len(p) for p in prompts)
    V = model.cfg.vocab_size
    input_ids = torch.full((B, L), PAD_ID, dtype=torch.long, device=dev)
    attn_mask = torch.zeros((B, L), dtype=torch.long, device=dev)
    for i, p in enumerate(prompts):                       # LEFT padding
        input_ids[i, L - len(p):] = torch.tensor(p, device=dev)
        attn_mask[i, L - len(p):] = 1

    amp_on = use_amp and dev.type == "cuda"
    always_ban = [t for t in (BOS_ID, PAD_ID, UNK_ID) if t is not None and 0 <= t < V]
    with torch.autocast(device_type="cuda", dtype=AMP_DTYPE, enabled=amp_on):
        logits, _, caches = model(input_ids, attention_mask=attn_mask, last_only=True)
    next_logits = logits[:, -1, :]

    generated = [[] for _ in range(B)]
    done = [False] * B                     # row has > min_words words
    seen = torch.zeros(B, V, dtype=torch.bool, device=dev)
    ngram_tables = [dict() for _ in range(B)] if no_repeat_ngram > 1 else None

    for step in range(max_new_tokens):
        lg = next_logits.float().clone()
        lg[:, always_ban] = -float("inf")
        not_done = torch.tensor([not d for d in done], device=dev)
        if EOS_ID is not None:
            lg[not_done, EOS_ID] = -float("inf")
        if repetition_penalty != 1.0:
            pen = torch.where(lg > 0, lg / repetition_penalty, lg * repetition_penalty)
            lg = torch.where(seen, pen, lg)
        if ngram_tables is not None:
            n = no_repeat_ngram
            for i in range(B):
                g = generated[i]
                if len(g) >= n - 1:
                    banned = ngram_tables[i].get(tuple(g[-(n - 1):]))
                    if banned:
                        lg[i, list(banned)] = -float("inf")
        if temperature == 0.0:
            nxt = lg.argmax(dim=-1)
        else:
            lg = lg / temperature
            if top_k > 0:
                kth = torch.topk(lg, top_k, dim=-1).values[:, -1:]
                lg = lg.masked_fill(lg < kth, -float("inf"))
            if top_p < 1.0:
                srt, idx = torch.sort(lg, descending=True, dim=-1)
                cdf = srt.softmax(-1).cumsum(-1)
                rm = cdf > top_p
                rm[:, 1:] = rm[:, :-1].clone()
                rm[:, 0] = False
                lg = lg.scatter(-1, idx, srt.masked_fill(rm, -float("inf")))
            nxt = torch.multinomial(lg.softmax(-1), 1).squeeze(-1)

        seen[torch.arange(B, device=dev), nxt] = True
        nxt_list = nxt.tolist()
        for i in range(B):
            generated[i].append(nxt_list[i])
            if ngram_tables is not None and len(generated[i]) >= no_repeat_ngram:
                g = generated[i]
                key = tuple(g[-no_repeat_ngram:-1])
                ngram_tables[i].setdefault(key, set()).add(g[-1])

        if (step + 1) % check_every == 0:
            for i in range(B):
                if not done[i] and _count_words(generated[i]) > min_words:
                    done[i] = True        # > min_words guarantees the 128th word is complete
            if all(done):
                break

        attn_mask = torch.cat([attn_mask, torch.ones(B, 1, dtype=torch.long, device=dev)], 1)
        with torch.autocast(device_type="cuda", dtype=AMP_DTYPE, enabled=amp_on):
            logits, _, caches = model(nxt.view(B, 1), attention_mask=attn_mask,
                                      kv_caches=caches, last_only=True)
        next_logits = logits[:, -1, :]

    out = []
    for g in generated:                 # cut anything after an EOS (only possible once done)
        out.append(g[:g.index(EOS_ID)] if EOS_ID in g else g)
    return out


def truncate_words(text: str, n: int = 128) -> str:
    return " ".join(text.split()[:n])


def continue_texts(model, texts: List[str], max_prompt_tokens: int = 768,
                   max_new_tokens: int = 256, batch_size: int = 16, verbose=False,
                   **decode_kw) -> List[str]:
    """Prompt strings -> 128-word continuation strings. Keeps the END of over-long prompts
    (the continuation must follow the prompt's last words)."""
    enc = []
    for t in texts:
        ids = tokenizer.encode(t, allow_special=False)
        enc.append(ids[-max_prompt_tokens:])
    order = sorted(range(len(texts)), key=lambda i: len(enc[i]))   # similar lengths per batch
    outs = [None] * len(texts)
    n_batches = (len(order) + batch_size - 1) // batch_size
    for bi in range(0, len(order), batch_size):
        idx = order[bi:bi + batch_size]
        gen = generate(model, [enc[i] for i in idx], max_new_tokens=max_new_tokens, **decode_kw)
        for i, g in zip(idx, gen):
            outs[i] = truncate_words(tokenizer.decode(g), 128)
        if verbose and (bi // batch_size) % 10 == 0:
            print(f"  batch {bi // batch_size + 1}/{n_batches}")
    return outs

In [15]:
# Cell 13 — Model tests (run on CPU or GPU in ~1 minute). Every one of these must pass.
#
# New vs model 01:
#  [3] "position mechanism is live": would have caught model 01's dropped RoPE.
#  [2] initial loss now uses SHIFTED targets (model 01 used targets=inputs, giving 8.9 not ~10.4).
#  [7] batched generation == one-at-a-time generation: catches padding/cache/inference bugs
#      of the kind that broke the encoder-decoder submission.
#  [8] MLA absorbed path == materialised path.  [9] ALiBi bias checked by hand.

def run_model_tests(cfg: ModelConfig, check_param_range=True, verbose=True):
    torch.manual_seed(0)
    dev = device
    model = LegalLM(cfg).to(dev)
    init_weights(model, cfg)
    model.eval()
    B, T = 2, 48
    ids = torch.randint(5, cfg.vocab_size, (B, T + 1), device=dev)
    x, y = ids[:, :-1], ids[:, 1:]
    log = (lambda *a: print(*a)) if verbose else (lambda *a: None)
    res = {}

    pc = count_params(model)
    log(f"[1] params total={pc['total']/1e6:.2f}M emb={pc['embedding']/1e6:.2f}M "
        f"non-emb={pc['non_embedding']/1e6:.2f}M attention={pc['attention']/1e6:.2f}M")
    if check_param_range:
        assert 50e6 <= pc["total"] <= 300e6, "param count outside 50M-300M"
    res["params"] = pc

    with torch.no_grad():
        full, loss, _ = model(x, targets=y)
    log(f"[2] initial loss {loss.item():.3f} (expected ~ln V = {math.log(cfg.vocab_size):.3f})")
    assert abs(loss.item() - math.log(cfg.vocab_size)) < 0.5

    if cfg.pos_type != "none":
        m2 = LegalLM(cfg).to(dev).eval()
        m2.load_state_dict(model.state_dict())
        with torch.no_grad():
            if cfg.pos_type == "alibi":
                m2.backbone.alibi.slopes.zero_()
            elif cfg.pos_type == "rope":
                m2.backbone.rope.cos.fill_(1.0)
                m2.backbone.rope.sin.zero_()
            else:
                m2.backbone.pos_emb.weight.zero_()
            lg_nopos, _, _ = m2(x)
        d = (lg_nopos - full).abs().max().item()
        log(f"[3] removing positional info changes logits by {d:.2e} (must be > 1e-3)")
        assert d > 1e-3, "positional mechanism is NOT wired in"

    with torch.no_grad():
        xb = x.clone()
        xb[:, -1] = (xb[:, -1] + 1) % cfg.vocab_size
        lb, _, _ = model(xb)
    d = (lb[:, :-1] - full[:, :-1]).abs().max().item()
    log(f"[4] causality: earlier logits change by {d:.2e}")
    assert d < 1e-5

    with torch.no_grad():
        half = T // 2
        lg_a, _, cache = model(x[:, :half])                  # chunked prefill
        steps = [lg_a]
        lg_b, _, cache = model(x[:, half:half + 5], kv_caches=cache)   # multi-token chunk
        steps.append(lg_b)
        for t in range(half + 5, T):                        # token-by-token
            lg_t, _, cache = model(x[:, t:t + 1], kv_caches=cache)
            steps.append(lg_t)
        d = (torch.cat(steps, 1) - full).abs().max().item()
    log(f"[5] KV-cache (chunked + stepwise) vs full: {d:.2e}")
    assert d < 1e-3
    res["kv_delta"] = d

    cfg_m = ModelConfig(**{**asdict(cfg), "use_sdpa": False})
    mm = LegalLM(cfg_m).to(dev).eval()
    mm.load_state_dict(model.state_dict())
    with torch.no_grad():
        lm, _, _ = mm(x)
    d = (lm - full).abs().max().item()
    log(f"[6a] SDPA vs manual attention: {d:.2e}")
    assert d < 1e-3

    Lp = 7
    padded = torch.cat([torch.full((1, Lp), PAD_ID, device=dev), x[:1]], 1)
    mask = torch.cat([torch.zeros(1, Lp, device=dev), torch.ones(1, T, device=dev)], 1).long()
    with torch.no_grad():
        lp, _, _ = model(padded, attention_mask=mask)
    d = (lp[0, Lp:] - full[0]).abs().max().item()
    log(f"[6b] left-padding equivalence: {d:.2e}  (finite: {torch.isfinite(lp).all().item()})")
    assert d < 1e-3 and torch.isfinite(lp).all()

    prompts = [x[0, :20].tolist(), x[1, :33].tolist(), x[0, 5:12].tolist()]
    kw = dict(max_new_tokens=12, min_words=10**9, check_every=1000, use_amp=False)
    batched = generate(model, prompts, **kw)
    single = [generate(model, [p], **kw)[0] for p in prompts]
    log(f"[7] batched generation == single generation: {batched == single}")
    assert batched == single

    if cfg.mixer_type == "mla":
        with torch.no_grad():
            model.set_mla_absorb(True)
            la, _, _ = model(x)
            lpa, _, _ = model(padded, attention_mask=mask)
            model.set_mla_absorb(False)
        d = max((la - full).abs().max().item(), (lpa[0, Lp:] - full[0]).abs().max().item())
        log(f"[8] MLA absorbed vs materialised: {d:.2e}")
        assert d < 1e-3

    if cfg.pos_type == "alibi":
        al = ALiBi(8)
        assert torch.allclose(al.slopes, torch.tensor([2.0 ** -(k + 1) for k in range(8)]))
        b = al.bias(4, 4, "cpu")[0]
        assert b[0, 3, 1].item() == -0.5 * 2          # head 1 slope 1/2, distance 2
        assert b[7, 3, 0].item() == -(1 / 256) * 3
        assert torch.isinf(b[:, 1, 2]).all()          # future is masked
        log("[9] ALiBi slopes/bias match hand calculation")
        with torch.no_grad():
            long_ids = torch.randint(5, cfg.vocab_size, (1, cfg.max_seq_len + 64), device=dev) \
                if dev.type == "cuda" else torch.randint(5, cfg.vocab_size, (1, 300), device=dev)
            lg_long, _, _ = model(long_ids)
        log(f"[10] runs past the training length: finite={torch.isfinite(lg_long).all().item()}")

    tmp = WORK_DIR / "_test_ckpt.pt"
    save_model(tmp, model, cfg)
    m3, _, _ = load_model(tmp, map_location=dev)
    with torch.no_grad():
        l3, _, _ = m3.to(dev).eval()(x)
    assert (l3 - full).abs().max().item() < 1e-5
    tmp.unlink()
    log("[11] checkpoint round-trip OK")
    log("ALL TESTS PASSED\n")
    del model, mm
    return res


for name, fn in PRESETS.items():
    print(f"=== {name} ===")
    run_model_tests(fn())

=== alibi_mla ===
[1] params total=53.89M emb=16.38M non-emb=37.50M attention=9.37M
[2] initial loss 10.476 (expected ~ln V = 10.373)
[3] removing positional info changes logits by 1.71e+00 (must be > 1e-3)
[4] causality: earlier logits change by 0.00e+00
[5] KV-cache (chunked + stepwise) vs full: 5.25e-06
[6a] SDPA vs manual attention: 3.34e-06
[6b] left-padding equivalence: 4.05e-06  (finite: True)
[7] batched generation == single generation: True
[8] MLA absorbed vs materialised: 4.53e-06
[9] ALiBi slopes/bias match hand calculation
[10] runs past the training length: finite=True
[11] checkpoint round-trip OK
ALL TESTS PASSED

=== alibi_mha ===
[1] params total=54.93M emb=16.38M non-emb=38.55M attention=12.58M
[2] initial loss 10.472 (expected ~ln V = 10.373)
[3] removing positional info changes logits by 1.67e+00 (must be > 1e-3)
[4] causality: earlier logits change by 0.00e+00
[5] KV-cache (chunked + stepwise) vs full: 2.92e-06
[6a] SDPA vs manual attention: 2.15e-06
[6b] left-pad

## Data and training

In [16]:
# Cell 14 — Token shards (streamed from HF, uint16 memmaps)
#
# FIX: model 01 used 150M tokens but trained on ~754M, i.e. ~5 passes over the same data.
# Caps are doubled here (300M tokens, SAME source proportions 13/43/17/27 %), so a faster fp16 run
# sees each token ~2x. Set DATA_SCALE = 1 to reproduce model 01's 150M exactly.
#
# Validation comparability: val docs are chosen by a hash of the doc id and appended in stream
# order, so a bigger cap only APPENDS val docs. The evaluator uses the first 64 windows x 1025
# tokens per source, which are therefore byte-identical to model 01's -> val losses comparable.
#
# Reuse: if shards already exist (in /kaggle/working or an attached Kaggle dataset), we skip
# encoding. Saving /kaggle/working/shards as a Kaggle dataset lets other runs reuse them.

import ijson
from array import array
from huggingface_hub import HfFileSystem
from tqdm.auto import tqdm

HF_REPO = "Exploration-Lab/CS-781-Capstone"
SOURCE_PATHS = {
    "acts":          "Phase_1_Pretraining/Train/train_acts.json",
    "high_courts":   "Phase_1_Pretraining/Train/train_high_courts.json",
    "supreme_court": "Phase_1_Pretraining/Train/train_supreme_court.json",
    "tribunals":     "Phase_1_Pretraining/Train/train_tribunals.json",
}
SOURCES = list(SOURCE_PATHS)
DATA_SCALE = 2
TOKEN_CAPS = {"acts": 20_000_000 * DATA_SCALE, "high_courts": 65_000_000 * DATA_SCALE,
              "supreme_court": 25_000_000 * DATA_SCALE, "tribunals": 40_000_000 * DATA_SCALE}
SHARD_TOKENS = 50_000_000
VAL_FRACTION = 0.005
VAL_CAP_TOTAL = 5_000_000

# Optional: point this at an attached Kaggle dataset containing <source>/meta.json + shards.
PREBUILT_SHARDS = Path("/kaggle/input/legal-lm-shards-300m")
SHARD_DIR = WORK_DIR / "shards"

def clean_text(t):
    return (t or "").replace("\r\n", "\n").strip()

def is_val(doc_id, salt="phase1"):
    h = int(hashlib.md5((salt + "::" + doc_id).encode()).hexdigest(), 16)
    return (h % 10_000) / 10_000.0 < VAL_FRACTION

def stream_docs(path_in_repo):
    fs = HfFileSystem()
    with fs.open(f"datasets/{HF_REPO}/{path_in_repo}", "rb") as f:
        for i, doc in enumerate(ijson.items(f, "item")):
            if isinstance(doc, dict):
                doc.setdefault("_idx", i)
                yield doc

def source_ready(root: Path, s: str) -> bool:
    mp = root / s / "meta.json"
    if not mp.exists():
        return False
    meta = json.loads(mp.read_text())
    return meta.get("cap", 0) >= TOKEN_CAPS[s]    # model-01 shards (no "cap" field) are re-encoded

def shards_ready(root: Path) -> bool:
    return all(source_ready(root, s) for s in SOURCES)

def encode_source(source, path_in_repo, out_dir):
    out_dir.mkdir(parents=True, exist_ok=True)
    meta = {"source": source, "hf_path": path_in_repo, "shards": [], "cap": TOKEN_CAPS[source],
            "total_tokens": 0, "total_bytes": 0, "num_docs": 0}
    buf, val_ids, shard_idx = array("H"), array("H"), 0
    val_cap, cap = VAL_CAP_TOTAL // len(SOURCE_PATHS), TOKEN_CAPS[source]

    def flush():
        nonlocal buf, shard_idx
        if len(buf) == 0:
            return
        arr = np.frombuffer(buf, dtype=np.uint16)
        final = out_dir / f"shard_{shard_idx:04d}.bin"
        tmp = final.with_suffix(".bin.tmp")
        arr.tofile(tmp); tmp.rename(final)
        meta["shards"].append({"name": final.name, "tokens": int(arr.size)})
        meta["total_tokens"] += int(arr.size)
        buf, shard_idx = array("H"), shard_idx + 1

    pbar = tqdm(desc=f"  {source}", unit="doc")
    for doc in stream_docs(path_in_repo):
        pbar.update(1)
        text = clean_text(doc.get("text"))
        if not text:
            continue
        doc_id = str(doc.get("url") or doc.get("id") or doc.get("_idx"))
        meta["num_docs"] += 1
        meta["total_bytes"] += len(text.encode("utf-8"))
        ids = tokenizer.encode(text, allow_special=False)
        if is_val(doc_id) and len(val_ids) < val_cap:
            val_ids.append(BOS_ID); val_ids.extend(ids); val_ids.append(EOS_ID)
            continue
        buf.append(BOS_ID); buf.extend(ids); buf.append(EOS_ID)
        if len(buf) >= SHARD_TOKENS:
            flush()
        if meta["total_tokens"] + len(buf) >= cap:
            break
    pbar.close()
    flush()
    if len(val_ids):
        (out_dir / "val.bin").write_bytes(np.frombuffer(val_ids, dtype=np.uint16).tobytes())
        meta["val_tokens"] = len(val_ids)
    (out_dir / "meta.json").write_text(json.dumps(meta, indent=2))
    print(f"[{source}] {meta['total_tokens']/1e6:.1f}M train tokens, "
          f"{meta.get('val_tokens', 0)/1e6:.2f}M val, {meta['num_docs']} docs")

if shards_ready(PREBUILT_SHARDS):
    SHARD_ROOT = PREBUILT_SHARDS
    print("using prebuilt shards:", SHARD_ROOT)
elif shards_ready(SHARD_DIR):
    SHARD_ROOT = SHARD_DIR
    print("using existing shards:", SHARD_ROOT)
else:
    t0 = time.time()
    for s, p in SOURCE_PATHS.items():    
        if not source_ready(SHARD_DIR, s):          # resumable per source
            encode_source(s, p, SHARD_DIR / s)
    SHARD_ROOT = SHARD_DIR
    print(f"encoded in {(time.time() - t0)/60:.1f} min")

  acts: 0doc [00:00, ?doc/s]

[acts] 40.0M train tokens, 0.15M val, 4891 docs


  high_courts: 0doc [00:00, ?doc/s]

[high_courts] 130.0M train tokens, 0.64M val, 217404 docs


  supreme_court: 0doc [00:00, ?doc/s]

[supreme_court] 50.0M train tokens, 0.23M val, 9935 docs


  tribunals: 0doc [00:00, ?doc/s]

[tribunals] 80.0M train tokens, 0.37M val, 28697 docs
encoded in 6.9 min


In [17]:
# !ls -lh
# !rm -rf shards/
# !ls -lh


In [18]:
# Cell 15 — Data loaders and evaluation

class ShardReader:
    def __init__(self, source_dir: Path):
        self.dir = Path(source_dir)
        meta = json.loads((self.dir / "meta.json").read_text())
        self.shards = meta["shards"]
        self.total_tokens = sum(s["tokens"] for s in self.shards)
        self.arrays = [np.memmap(self.dir / s["name"], dtype=np.uint16, mode="r") for s in self.shards]
        w = np.array([s["tokens"] for s in self.shards], dtype=np.float64)
        self.shard_weights = w / w.sum()
        vp = self.dir / "val.bin"
        self.val_arr = np.memmap(vp, dtype=np.uint16, mode="r") if vp.exists() else None


def build_readers(root: Path) -> Dict[str, "ShardReader"]:
    return {s: ShardReader(root / s) for s in SOURCES if (root / s / "meta.json").exists()}


class MixedWindowLoader:
    """Random windows; the batch is a pure function of (seed, step) -> exact resume."""
    def __init__(self, readers, weights: Dict[str, float], seq_len, micro_batch, seed=0):
        self.readers, self.seq_len, self.micro_batch, self.seed = readers, seq_len, micro_batch, seed
        self.keys = list(readers)
        p = np.array([weights[k] for k in self.keys], dtype=np.float64)
        self.p = p / p.sum()

    def get_batch(self, step: int):
        rng = np.random.default_rng((self.seed * 1_000_003 + step * 2_654_435_761) & 0xFFFFFFFF)
        B, T = self.micro_batch, self.seq_len
        out = np.zeros((B, T + 1), dtype=np.int64)
        for b in range(B):
            r = self.readers[rng.choice(self.keys, p=self.p)]
            arr = r.arrays[rng.choice(len(r.arrays), p=r.shard_weights)]
            start = int(rng.integers(0, max(1, len(arr) - (T + 1))))
            w = np.asarray(arr[start:start + T + 1], dtype=np.int64)
            out[b, :len(w)] = w
        t = torch.from_numpy(out)
        return t[:, :-1], t[:, 1:]


class FixedValLoader:
    """First `windows_per_source` non-overlapping (seq_len+1)-token windows of each val.bin.
    With seq_len=1024, windows_per_source=64 this is exactly model 01's validation set."""
    def __init__(self, readers, seq_len=1024, windows_per_source=64, batch_size=8):
        self.seq_len, self.batch_size = seq_len, batch_size
        self.windows = []
        for name, r in readers.items():
            if r.val_arr is None:
                continue
            n = min(windows_per_source, len(r.val_arr) // (seq_len + 1))
            for i in range(n):
                s = i * (seq_len + 1)
                self.windows.append((name, np.asarray(r.val_arr[s:s + seq_len + 1], dtype=np.int64)))

    def __iter__(self):
        for i in range(0, len(self.windows), self.batch_size):
            chunk = self.windows[i:i + self.batch_size]
            arr = torch.from_numpy(np.stack([w for _, w in chunk]))
            yield [n for n, _ in chunk], arr[:, :-1], arr[:, 1:]


@torch.no_grad()
def evaluate(model, val_loader, last_k: Optional[int] = None):
    """Mean token loss overall and per source (exact, from per-sequence losses).
    last_k: score only the last k targets of each window (used for context-length studies)."""
    model.eval()
    sums, counts = {}, {}
    for srcs, x, y in val_loader:
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        with torch.autocast(device_type="cuda", dtype=AMP_DTYPE, enabled=USE_AMP):
            _, loss_tok, _ = model(x, targets=y, reduction="none")       # [B, T]
        if last_k is not None:
            loss_tok = loss_tok[:, -last_k:]
        per_seq = loss_tok.float().sum(1).tolist()
        n = loss_tok.shape[1]
        for s, l in zip(srcs, per_seq):
            sums[s] = sums.get(s, 0.0) + l
            counts[s] = counts.get(s, 0) + n
    model.train()
    per_src = {s: sums[s] / counts[s] for s in sums}
    overall = sum(sums.values()) / sum(counts.values())
    return overall, per_src

In [19]:
# Cell 16 — Training loop
#
# Same optimiser recipe as model 01 (AdamW 0.9/0.95, wd 0.1, peak 6e-4 -> 6e-5 cosine, clip 1.0,
# ~131k tokens/step) so the comparison is controlled. Changes: fp16 autocast + GradScaler on T4,
# exact per-source eval, and total_steps derived from MEASURED throughput (next cell) so the cosine
# schedule actually finishes inside the time budget (model 01 guessed 6000 and stopped at ~5800).

@dataclass
class TrainConfig:
    seq_len: int = 1024
    micro_batch: int = 16
    grad_accum: int = 8
    peak_lr: float = 6e-4
    min_lr: float = 6e-5
    warmup_steps: int = 400
    # warmup_steps: int = 10
    total_steps: int = 6000
    # total_steps: int = 40
    weight_decay: float = 0.1
    betas: Tuple[float, float] = (0.9, 0.95)
    eps: float = 1e-8
    grad_clip: float = 1.0
    eval_every: int = 250
    # eval_every: int = 10
    log_every: int = 50
    # log_every: int = 5
    ckpt_every_minutes: float = 20
    # ckpt_every_minutes: float = 2
    time_budget_hours: float = 8.5
    # time_budget_hours: float = 0.1
    seed: int = 1337
    output_dir: str = "runs/alibi_mla"

    @property
    def tokens_per_step(self):
        return self.seq_len * self.micro_batch * self.grad_accum


def cosine_lr(step, warmup, total, peak, minimum):
    if step < warmup:
        return peak * (step + 1) / warmup
    if step >= total:
        return minimum
    prog = (step - warmup) / max(1, total - warmup)
    return minimum + 0.5 * (peak - minimum) * (1 + math.cos(math.pi * prog))


def train(cfg: ModelConfig, tcfg: TrainConfig, train_loader, val_loader):
    out_dir = Path(tcfg.output_dir); out_dir.mkdir(parents=True, exist_ok=True)
    log_path = out_dir / "log.jsonl"
    model = LegalLM(cfg).to(device)
    init_weights(model, cfg)
    opt = torch.optim.AdamW(build_param_groups(model, tcfg.weight_decay), lr=tcfg.peak_lr,
                            betas=tcfg.betas, eps=tcfg.eps)
    scaler = torch.amp.GradScaler("cuda", enabled=USE_SCALER)

    start_step, best_val = 0, float("inf")
    last = out_dir / "last.pt"
    if last.exists():                                   # automatic resume
        ck = torch.load(last, map_location=device, weights_only=False)
        model.load_state_dict(ck["model"]); opt.load_state_dict(ck["opt"])
        scaler.load_state_dict(ck["scaler"])
        start_step, best_val = ck["step"] + 1, ck.get("best_val", float("inf"))
        print(f"resumed from step {ck['step']} (best val {best_val:.4f})")
    (out_dir / "config.json").write_text(json.dumps({"model": asdict(cfg), "train": asdict(tcfg)},
                                                    indent=2))
    pc = count_params(model)
    print(f"params: {pc['total']/1e6:.2f}M | tokens/step {tcfg.tokens_per_step:,} "
          f"| steps {tcfg.total_steps}")

    model.train()
    t0 = last_ckpt = time.time()
    tokens_seen = 0
    step = start_step
    for step in range(start_step, tcfg.total_steps):
        lr = cosine_lr(step, tcfg.warmup_steps, tcfg.total_steps, tcfg.peak_lr, tcfg.min_lr)
        for g in opt.param_groups:
            g["lr"] = lr
        opt.zero_grad(set_to_none=True)
        step_loss = 0.0
        for micro in range(tcfg.grad_accum):
            x, y = train_loader.get_batch(step * tcfg.grad_accum + micro)
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            with torch.autocast(device_type="cuda", dtype=AMP_DTYPE, enabled=USE_AMP):
                _, loss, _ = model(x, targets=y)
            scaler.scale(loss / tcfg.grad_accum).backward()
            step_loss += loss.item() / tcfg.grad_accum
            tokens_seen += y.numel()
        scaler.unscale_(opt)
        gn = torch.nn.utils.clip_grad_norm_(model.parameters(), tcfg.grad_clip)
        scaler.step(opt); scaler.update()

        if step % tcfg.log_every == 0:
            dt = time.time() - t0
            rec = {"step": step, "loss": step_loss, "lr": lr, "grad_norm": float(gn),
                   "tokens_per_s": tokens_seen / max(dt, 1e-6), "elapsed_s": dt,
                   "scale": scaler.get_scale() if USE_SCALER else 1.0}
            with open(log_path, "a") as f:
                f.write(json.dumps(rec) + "\n")
            print(f"step {step:5d} | loss {step_loss:.4f} | lr {lr:.2e} | gnorm {float(gn):.2f} "
                  f"| {rec['tokens_per_s']/1e3:.1f}k tok/s | {dt/60:.1f}m")

        is_last = (step == tcfg.total_steps - 1)
        out_of_time = (time.time() - t0) / 3600 >= tcfg.time_budget_hours
        if (step > 0 and step % tcfg.eval_every == 0) or is_last or out_of_time:
            vl, per = evaluate(model, val_loader)
            print(f"  [eval] step {step} val_loss={vl:.4f} ppl={math.exp(vl):.2f} "
                  f"per_src={ {k: round(v, 3) for k, v in per.items()} }")
            with open(log_path, "a") as f:
                f.write(json.dumps({"step": step, "val_loss": vl, "val_per_src": per}) + "\n")
            if vl < best_val:
                best_val = vl
                save_model(out_dir / "best.pt", model, cfg, extra={"step": step, "val_loss": vl})

        if (time.time() - last_ckpt) / 60 >= tcfg.ckpt_every_minutes or out_of_time:
            torch.save({"model": model.state_dict(), "opt": opt.state_dict(),
                        "scaler": scaler.state_dict(), "step": step, "best_val": best_val,
                        "config": asdict(cfg)}, out_dir / "last.pt.tmp")
            os.replace(out_dir / "last.pt.tmp", last)          # atomic: never a half-written file
            last_ckpt = time.time()
            print(f"  [ckpt] step {step}")
        if out_of_time:
            print(f"time budget reached at step {step}")
            break

    save_model(out_dir / "final.pt", model, cfg, extra={"step": step})
    return model, out_dir

In [20]:
# Cell 17 — Memory probe + throughput benchmark -> micro-batch and schedule length
#
# Picks the largest micro-batch that fits (keeping tokens/step = 131,072 via grad_accum), measures
# real tokens/s, and sizes total_steps so warmup + cosine decay end just inside the time budget.

ARCH = "alibi_mla"                 # "alibi_mla" (main) | "alibi_mha" (ablation) | "rope_mha" (baseline)
TIME_BUDGET_HOURS = 8.5            # leave ~1.5 h of the 12 h session for data, tests, submission
TOKENS_PER_STEP = 131_072          # same as model 01

cfg = PRESETS[ARCH]()
readers = build_readers(SHARD_ROOT)
sizes = {k: r.total_tokens for k, r in readers.items()}
mix = {k: v / sum(sizes.values()) for k, v in sizes.items()}     # proportional, as model 01
print("train tokens:", {k: f"{v/1e6:.0f}M" for k, v in sizes.items()}, "| mix:",
      {k: round(v, 3) for k, v in mix.items()})
val_loader = FixedValLoader(readers, seq_len=1024, windows_per_source=64, batch_size=8)
print("val windows:", len(val_loader.windows))

def try_micro_batch(mb, n_timed=0):
    model = LegalLM(cfg).to(device)
    init_weights(model, cfg)
    opt = torch.optim.AdamW(model.parameters(), lr=1e-4)
    scaler = torch.amp.GradScaler("cuda", enabled=USE_SCALER)
    loader = MixedWindowLoader(readers, mix, 1024, mb, seed=99)
    times = []
    try:
        for i in range(2 + n_timed):
            x, y = loader.get_batch(i)
            if device.type == "cuda":
                torch.cuda.synchronize()
            t = time.time()
            with torch.autocast(device_type="cuda", dtype=AMP_DTYPE, enabled=USE_AMP):
                _, loss, _ = model(x.to(device), targets=y.to(device))
            scaler.scale(loss).backward()
            scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True)
            if device.type == "cuda":
                torch.cuda.synchronize()
            if i >= 2:
                times.append(time.time() - t)
        peak = torch.cuda.max_memory_allocated() / 2**30 if device.type == "cuda" else 0
        return True, (mb * 1024 / np.mean(times) if times else None), peak
    except torch.OutOfMemoryError:
        return False, None, None
    finally:
        del model, opt
        gc.collect()
        if device.type == "cuda":
            torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()

micro = None
for mb in (32, 16, 8, 4):
    ok, _, peak = try_micro_batch(mb)
    print(f"micro_batch={mb}: {'fits' if ok else 'OOM'}" + (f" (peak {peak:.1f} GB)" if ok else ""))
    if ok:
        micro = mb
        break
assert micro is not None
_, tok_s, _ = try_micro_batch(micro, n_timed=6)
steps = int(TIME_BUDGET_HOURS * 3600 * tok_s * 0.95 / TOKENS_PER_STEP)   # 5% for eval/ckpt

# steps = 40 # mini test run 

print(f"throughput ~{tok_s/1e3:.1f}k tok/s -> {steps} steps "
      f"(~{steps * TOKENS_PER_STEP / 1e6:.0f}M tokens, "
      f"~{steps * TOKENS_PER_STEP / sum(sizes.values()):.1f} passes over the data)")

tcfg = TrainConfig(micro_batch=micro, grad_accum=TOKENS_PER_STEP // (1024 * micro),
                   total_steps=steps, warmup_steps=400 if steps >= 3000 else max(50, steps // 15),
                   time_budget_hours=TIME_BUDGET_HOURS,
                   output_dir=str(WORK_DIR / "runs" / ARCH))
train_loader = MixedWindowLoader(readers, mix, tcfg.seq_len, tcfg.micro_batch, seed=SEED)
print(tcfg)

train tokens: {'acts': '40M', 'high_courts': '130M', 'supreme_court': '50M', 'tribunals': '80M'} | mix: {'acts': 0.133, 'high_courts': 0.433, 'supreme_court': 0.167, 'tribunals': 0.267}
val windows: 256
micro_batch=32: OOM
micro_batch=16: fits (peak 12.8 GB)
throughput ~21.0k tok/s -> 4656 steps (~610M tokens, ~2.0 passes over the data)
TrainConfig(seq_len=1024, micro_batch=16, grad_accum=8, peak_lr=0.0006, min_lr=6e-05, warmup_steps=400, total_steps=4656, weight_decay=0.1, betas=(0.9, 0.95), eps=1e-08, grad_clip=1.0, eval_every=250, log_every=50, ckpt_every_minutes=20, time_budget_hours=8.5, seed=1337, output_dir='/kaggle/working/runs/alibi_mla')


In [21]:
# Cell 18 — Full training run (~TIME_BUDGET_HOURS). Re-running this cell resumes from last.pt.
model, run_dir = train(cfg, tcfg, train_loader, val_loader)

params: 53.89M | tokens/step 131,072 | steps 4656
step     0 | loss 10.4072 | lr 1.50e-06 | gnorm 7.50 | 20.9k tok/s | 0.1m
step    50 | loss 6.9878 | lr 7.65e-05 | gnorm 2.60 | 19.7k tok/s | 5.7m
step   100 | loss 5.0188 | lr 1.51e-04 | gnorm 2.19 | 19.7k tok/s | 11.2m
step   150 | loss 4.0443 | lr 2.26e-04 | gnorm 1.65 | 19.7k tok/s | 16.8m
  [ckpt] step 179
step   200 | loss 3.5188 | lr 3.01e-04 | gnorm 1.65 | 19.6k tok/s | 22.4m
step   250 | loss 3.2040 | lr 3.76e-04 | gnorm 1.01 | 19.7k tok/s | 27.9m
  [eval] step 250 val_loss=3.5713 ppl=35.56 per_src={'acts': 4.061, 'high_courts': 2.224, 'supreme_court': 4.277, 'tribunals': 3.724}
step   300 | loss 2.8505 | lr 4.51e-04 | gnorm 0.85 | 19.6k tok/s | 33.5m
step   350 | loss 2.8223 | lr 5.26e-04 | gnorm 0.71 | 19.6k tok/s | 39.1m
  [ckpt] step 359
step   400 | loss 2.7280 | lr 6.00e-04 | gnorm 0.71 | 19.6k tok/s | 44.6m
step   450 | loss 2.6725 | lr 6.00e-04 | gnorm 0.51 | 19.6k tok/s | 50.2m
step   500 | loss 2.7519 | lr 5.99e-04 | 

## Selection, submission and analysis

In [22]:
# Cell 19 — Local dev set: checkpoint selection + decoding choice on the REAL task format
#
# Built from our held-out val documents: 512 words of prompt -> the next 128 words as reference.
# Scored with word-level ROUGE-L F1 and BLEU-4 (the competition's secondary metrics; the primary,
# BERTScore, needs the official evaluation scripts). Lets us choose decoding settings and
# best.pt vs final.pt on generation quality, not on val loss alone.

def build_dev_set(readers, per_source=25, prompt_words=512, ref_words=128, seed=0):
    rng = np.random.default_rng(seed)
    dev = []
    for name, r in readers.items():
        arr = np.asarray(r.val_arr, dtype=np.int64)
        bos = np.where(arr == BOS_ID)[0].tolist() + [len(arr)]
        docs = []
        for a, b in zip(bos[:-1], bos[1:]):
            words = tokenizer.decode(arr[a + 1:b].tolist()).split()
            if len(words) >= prompt_words + ref_words:
                docs.append(words)
        rng.shuffle(docs)
        for words in docs[:per_source]:
            s = int(rng.integers(0, len(words) - (prompt_words + ref_words) + 1))
            dev.append({"source": name, "prompt": " ".join(words[s:s + prompt_words]),
                        "ref": " ".join(words[s + prompt_words:s + prompt_words + ref_words])})
    return dev


def rouge_l_f1(hyp: str, ref: str) -> float:
    h, r = hyp.split(), ref.split()
    if not h or not r:
        return 0.0
    prev = [0] * (len(r) + 1)
    for hw in h:                              # O(|h||r|) longest common subsequence
        cur = [0] * (len(r) + 1)
        for j, rw in enumerate(r, 1):
            cur[j] = prev[j - 1] + 1 if hw == rw else max(prev[j], cur[j - 1])
        prev = cur
    lcs = prev[-1]
    if lcs == 0:
        return 0.0
    p, rc = lcs / len(h), lcs / len(r)
    return 2 * p * rc / (p + rc)


def corpus_bleu4(hyps: List[str], refs: List[str]) -> float:
    from collections import Counter
    match, total = [0] * 4, [0] * 4
    hl = rl = 0
    for hyp, ref in zip(hyps, refs):
        h, r = hyp.split(), ref.split()
        hl, rl = hl + len(h), rl + len(r)
        for n in range(1, 5):
            hc = Counter(tuple(h[i:i + n]) for i in range(len(h) - n + 1))
            rc = Counter(tuple(r[i:i + n]) for i in range(len(r) - n + 1))
            match[n - 1] += sum(min(c, rc[g]) for g, c in hc.items())
            total[n - 1] += max(len(h) - n + 1, 0)
    if min(match) == 0:
        return 0.0
    logp = sum(math.log(m / t) for m, t in zip(match, total)) / 4
    bp = 1.0 if hl > rl else math.exp(1 - rl / max(hl, 1))
    return bp * math.exp(logp)


def score(model, dev, decode_kw, batch_size=16):
    hyps = continue_texts(model, [d["prompt"] for d in dev], batch_size=batch_size, **decode_kw)
    refs = [d["ref"] for d in dev]
    rl = float(np.mean([rouge_l_f1(h, r) for h, r in zip(hyps, refs)]))
    return {"rougeL": rl, "bleu4": corpus_bleu4(hyps, refs),
            "mean_words": float(np.mean([len(h.split()) for h in hyps])), "sample": hyps[0][:300]}


dev = build_dev_set(readers)
print(len(dev), "dev examples:", {s: sum(d["source"] == s for d in dev) for s in readers})

DECODE_CONFIGS = {
    "A": dict(temperature=0.0, repetition_penalty=1.1),                      # model 01 setting
    "B": dict(temperature=0.0, repetition_penalty=1.1, no_repeat_ngram=4),
    "C": dict(temperature=0.0, repetition_penalty=1.0, no_repeat_ngram=3),
    "D": dict(temperature=0.7, top_p=0.9, repetition_penalty=1.1),
}

CKPT_MODELS = {}
for tag in ("best", "final"):
    path = WORK_DIR / "runs" / ARCH / f"{tag}.pt"
    if path.exists():
        m, c, ex = load_model(path, map_location=device)
        m = m.to(device).eval()
        if c.mixer_type == "mla":
            m.set_mla_absorb(True)
        CKPT_MODELS[tag] = m
        print(f"{tag}.pt: step {ex.get('step')}, val_loss {ex.get('val_loss')}")

torch.manual_seed(0)
results = {}
for name, kw in DECODE_CONFIGS.items():                 # decoding sweep on best.pt
    results[("best", name)] = score(CKPT_MODELS["best"], dev, kw)
if "final" in CKPT_MODELS:                              # checkpoint comparison with config A
    results[("final", "A")] = score(CKPT_MODELS["final"], dev, DECODE_CONFIGS["A"])

print(f"\n{'ckpt':6s} {'cfg':4s} {'ROUGE-L':>8s} {'BLEU-4':>8s} {'words':>6s}")
for (ck, name), r in results.items():
    print(f"{ck:6s} {name:4s} {r['rougeL']:8.4f} {r['bleu4']:8.4f} {r['mean_words']:6.1f}")
(BEST_CKPT, best_name) = max(results, key=lambda k: results[k]["rougeL"])
if BEST_CKPT == "final" and ("final", best_name) not in results:
    BEST_CKPT = "best"
BEST_DECODE = DECODE_CONFIGS[best_name]
print(f"\nchosen: {BEST_CKPT}.pt + config {best_name} {BEST_DECODE}")
print("sample:", results[(BEST_CKPT, best_name)]["sample"])
(WORK_DIR / "runs" / ARCH / "dev_results.json").write_text(json.dumps(
    {f"{k[0]}_{k[1]}": v for k, v in results.items()}, indent=2))

92 dev examples: {'acts': 17, 'high_courts': 25, 'supreme_court': 25, 'tribunals': 25}
best.pt: step 4591, val_loss 1.8845605307433289
final.pt: step 4591, val_loss None

ckpt   cfg   ROUGE-L   BLEU-4  words
best   A      0.3100   0.2245  125.7
best   B      0.3074   0.2165  127.7
best   C      0.2978   0.1968  126.8
best   D      0.3038   0.2200  127.8
final  A      0.3100   0.2245  125.7

chosen: best.pt + config A {'temperature': 0.0, 'repetition_penalty': 1.1}
sample: it is necessary to appoint or designate an officer of the Commission, it may appoint an officer of the Commission, who shall be responsible for the conduct of its business. (10) The Ombudsman shall be a person who has been appointed as Ombudsman and shall have the powers of a Civil Court under the C


2204

In [23]:
# Cell 20 — Codabench submission (checkpoint + decoding chosen on the local dev set in Cell 19)

from datasets import load_dataset

def load_test_prompts_hf(max_n=None) -> Dict[str, str]:
    """The phase_1 test split is one HF row whose columns are the ids '1'..'1000'."""
    ds = load_dataset("Exploration-Lab/CS-781-Capstone", "phase_1", split="test",
                      streaming=True, token=True)
    out = {}
    for row in ds:
        for k, v in row.items():
            out[str(k)] = v.get("text", "") if isinstance(v, dict) else v
            if max_n is not None and len(out) >= max_n:
                return out
    return out


def make_submission(model, prompts: Dict[str, str], out_dir: Path, **decode_kw):
    ids = list(prompts)
    t0 = time.time()
    texts = continue_texts(model, [prompts[k] for k in ids], verbose=True, **decode_kw)
    preds = {k: {"text": t} for k, t in zip(ids, texts)}
    wc = [len(t.split()) for t in texts]
    print(f"done in {(time.time() - t0)/60:.1f} min | words min/mean/max = "
          f"{min(wc)}/{np.mean(wc):.1f}/{max(wc)} | short (<128): {sum(w < 128 for w in wc)}")
    assert len(preds) == len(prompts)
    out_dir = Path(out_dir); out_dir.mkdir(parents=True, exist_ok=True)
    (out_dir / "prediction.json").write_text(json.dumps(preds, ensure_ascii=False))
    with zipfile.ZipFile(out_dir / "submission.zip", "w", zipfile.ZIP_DEFLATED) as z:
        z.write(out_dir / "prediction.json", arcname="prediction.json")
    print("wrote", out_dir / "submission.zip")
    return preds


sub_model = CKPT_MODELS[BEST_CKPT]
print(f"submitting {ARCH} / {BEST_CKPT} with decoding {BEST_DECODE}")
prompts = load_test_prompts_hf()
print(len(prompts), "prompts | sample:", next(iter(prompts.values()))[:100], "...")
# Prompt budget: 768 tokens (~540 words) holds the full 512-word prompt for almost all inputs;
# 768 + 256 new tokens = 1024 = training length.
preds = make_submission(sub_model, prompts, WORK_DIR / "submission" / ARCH,
                        max_prompt_tokens=768, max_new_tokens=256, batch_size=16, **BEST_DECODE)

# Extra Codabench submissions (Phase 1 needs >= 4) are cheap: re-run make_submission with another
# row of DECODE_CONFIGS into a different folder, e.g.
# make_submission(sub_model, prompts, WORK_DIR / "submission" / f"{ARCH}_cfgB", **DECODE_CONFIGS["B"])

submitting alibi_mla / best with decoding {'temperature': 0.0, 'repetition_penalty': 1.1}


README.md:   0%|          | 0.00/16.1k [00:00<?, ?B/s]

1000 prompts | sample: The issue involved in these petitions is as to whether the acquisition proceedings can be challenged ...
  batch 1/63
  batch 11/63
  batch 21/63
  batch 31/63
  batch 41/63
  batch 51/63
  batch 61/63
done in 4.9 min | words min/mean/max = 2/126.6/128 | short (<128): 58
wrote /kaggle/working/submission/alibi_mla/submission.zip


In [24]:
# Cell 21 — Analysis for the report: context length / ALiBi extrapolation, and KV-cache size
#
# (a) Loss on the LAST 256 tokens of windows of growing length. Trained at 1024: if the loss keeps
#     falling at 1536/2048 the model is using context beyond its training length (ALiBi's claim);
#     a RoPE model typically degrades past 1024. Run the same cell on each architecture.
# (b) KV-cache memory per sequence at inference: MHA vs MLA.

ana_model = CKPT_MODELS["best"]
ana_model.set_mla_absorb(False)
rows = []
for L in (512, 1024, 1536, 2048):
    if cfg.pos_type == "rope" and L > 2 * cfg.max_seq_len:
        continue
    vl = FixedValLoader(readers, seq_len=L, windows_per_source=24, batch_size=2)
    loss, _ = evaluate(ana_model, vl, last_k=256)
    rows.append((L, loss))
    print(f"context {L:5d}: loss on last 256 tokens = {loss:.4f} (ppl {math.exp(loss):.2f})")

bytes_per = 2   # fp16 cache
for name in ("rope_mha", "alibi_mla"):
    c = PRESETS[name]()
    per_tok = (2 * c.n_kv_head * c.head_dim) if c.mixer_type == "attention" else c.kv_latent_dim
    mb = c.n_layer * per_tok * 1024 * bytes_per / 2**20
    print(f"{name:10s}: {per_tok:5d} cached numbers/token/layer -> {mb:6.2f} MB per 1024-token sequence")

context   512: loss on last 256 tokens = 1.7484 (ppl 5.75)
context  1024: loss on last 256 tokens = 1.7876 (ppl 5.98)
context  1536: loss on last 256 tokens = 1.7358 (ppl 5.67)
context  2048: loss on last 256 tokens = 1.6717 (ppl 5.32)
rope_mha  :  1024 cached numbers/token/layer ->  24.00 MB per 1024-token sequence
alibi_mla :   128 cached numbers/token/layer ->   3.25 MB per 1024-token sequence
